# Context Engineering Fundamentals
## Building Better Prompts with Layered Context

In this notebook, we'll explore how to structure context at multiple levels to improve LLM outputs. We'll demonstrate:
- **System-level context** (role definition)
- **Domain context** (business information)
- **Environment context** (market conditions)
- **Session context** (conversation history)
- **Prompt-level context** (specific instructions)

## Step 1: Initialize the LLM

First, we set up our language model with OpenAI's GPT-4o. The API key is loaded from environment variables for security.

**Key Configuration:**
- **Model**: GPT-4o (latest, most capable model)
- **Temperature**: 0 (deterministic responses)
- **API Key**: Loaded from `.env` file

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# Load environment variables from .env file
load_dotenv()

# Initialize LLM with key from environment
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY")
)

## Step 2: System & Domain Context

We now demonstrate the **first two levels of context**:

1. **System Context**: Defines the model's role and behavior
   - Establishes expertise level
   - Sets tone and style
   - Guides response format

2. **Domain Context**: Provides business-specific information
   - Product details
   - Market positioning
   - Competitive landscape

These combine to create a focused prompt that guides the model toward relevant, practical responses.

In [ ]:
# User prompt
prompt = "Recommend the best pricing strategy for launching our AI product."

# System-Level Context (Role)
system_context = """You are a senior product strategist.
Provide structured, concise, and practical recommendations.
"""

# Domain Context (Business info)
domain_context = """
The product is a B2B AI SaaS platform sold to mid-size enterprises.
Competitors use usage-based and tiered pricing.
"""

# Combine everything into messages (THIS is the correct pattern)
messages = [
    SystemMessage(content=system_context),
    HumanMessage(content=f"""
Context:
{domain_context}

Question:
{prompt}
""")
]

# Invoke model
response = llm.invoke(messages)

print(response.content)

## Step 3: Adding Environment Context

Now we enhance the prompt with **Environment Context** - information about market conditions, customer behavior, and external factors.

**What's included:**
- Market competition intensity
- Customer buying patterns
- Sales cycle timelines
- Procurement requirements

Notice how each additional context layer helps the model provide more nuanced and contextually appropriate recommendations.

In [ ]:
environment_context = """
Market conditions:
- High competition
- Customers are cost-sensitive
- Sales cycle is 3–6 months
- Procurement requires predictable pricing
"""

combined_context = domain_context + environment_context

messages = [
    SystemMessage(content=system_context),
    HumanMessage(content=f"""
Context:
{combined_context}

Question:
{prompt}
""")
]

response = llm.invoke(messages)
print(response.content)

## Step 4: Adding Session Context

We now layer in **Session Context** - information from previous discussions and interactions that inform the current decision.

**What's included:**
- Previous customer feedback
- Known preferences and constraints
- Earlier discussion points

This demonstrates how conversation history shapes recommendations.

In [ ]:
session_context = """
Earlier discussion:
- Target customers distrust pure usage-based pricing
- CFOs prefer capped monthly spend
"""

combined_context = (
    domain_context +
    environment_context +
    session_context
)

messages = [
    SystemMessage(content=system_context),
    HumanMessage(content=f"""
Context:
{combined_context}

Question:
{prompt}
""")
]

response = llm.invoke(messages)
print(response.content)

## Step 5: Adding Prompt-Level Context

Finally, we add **Prompt-Level Context** - specific instructions and focus areas for this particular query.

**What's included:**
- Specific scenario details (launch phase)
- Key priorities (buyer friction, value clarity)
- Decision factors to emphasize

This is the most granular level of context, tailoring the response to the specific moment and need.

In [ ]:
prompt_level_context = """
Focus on:
- Initial launch phase
- Reducing buyer friction
- Clear value communication
"""

final_context = (
    domain_context +
    environment_context +
    session_context +
    prompt_level_context
)

messages = [
    SystemMessage(content=system_context),
    HumanMessage(content=f"""
Context:
{final_context}

Question:
{prompt}
""")
]

response = llm.invoke(messages)
print(response.content)

## Key Takeaways

### The Context Pyramid

Context engineering involves layering information at different levels:

```
┌─────────────────────────────────────┐
│    Prompt-Level Context             │  Most Specific
│    (Immediate task focus)           │
├─────────────────────────────────────┤
│    Session Context                  │
│    (Conversation history)           │
├─────────────────────────────────────┤
│    Environment Context              │
│    (Market & External factors)      │
├─────────────────────────────────────┤
│    Domain Context                   │
│    (Business information)           │
├─────────────────────────────────────┤
│    System Context                   │
│    (Role & Behavior)                │  Most General
└─────────────────────────────────────┘
```

### Best Practices

1. **Be Specific**: More specific context leads to better responses
2. **Stay Relevant**: Include only context that affects the decision
3. **Use Structure**: Organize context hierarchically
4. **Update Dynamically**: Add session and prompt context as needed
5. **Test & Iterate**: Experiment with different context combinations